# 0.4 - Baseline Models

Untuned baselines trained on log1p(radon), evaluated on the test subset (log-scale regression metrics, average precision at the reference level).

## Setup

In [ ]:
import pickle

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import average_precision_score  
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

from evaluation import load_test_set, regression_scores

ROOT = '../../'
DIR = '../../models/'
RANDOM_SEED = 42
RADON_REFERENCE_LEVEL = 300  # Bq/m3, for the average precision

## Data

In [ ]:
X_train = pd.read_csv(f'{ROOT}data/final/dataset-v0.2-x-train.csv', index_col=0).replace([np.inf, -np.inf], np.nan)
y_train = pd.read_csv(f'{ROOT}data/final/dataset-v0.2-y-train.csv', index_col=0).squeeze()
X_test, y_test = load_test_set(ROOT)

with open(f'{DIR}0.2-encoding-pipeline.pkl', 'rb') as file:
    preprocessor = pickle.load(file)
X_train_enc = preprocessor.transform(X_train)
X_test_enc = preprocessor.transform(X_test)

## Models

In [ ]:
results = []


def evaluate(name, y_pred_log):
    y_pred = np.expm1(y_pred_log)
    results.append({
        'Model': name,
        **regression_scores(y_test, y_pred),
        'Average precision': round(average_precision_score(y_test >= RADON_REFERENCE_LEVEL, y_pred), 3),
    })


models = {
    'Linear regression': LinearRegression(), 
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.001),
    'Elastic net': ElasticNet(alpha=0.001, l1_ratio=0.5),
    'Decision tree': DecisionTreeRegressor(max_depth=5, random_state=RANDOM_SEED),
    'MLP': MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=RANDOM_SEED),
    'Random forest': RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED),
    'Gradient boosting': GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_SEED),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=RANDOM_SEED),
}
for name, model in models.items():
    model.fit(X_train_enc, np.log1p(y_train))
    evaluate(name, model.predict(X_test_enc))

HistGradientBoosting with native categorical support, on a hand-picked feature subset:

In [ ]:
numerical_features = [
    'Rdn - Année de construction',
    #'Rdn - Heures de séjour par an',
    #'Rdn - duration_days',
    #'RegBL - Surface du bâtiment',
    #'RegBL - Nombre de niveaux',
    #'RegBL - Nombre de logements',
    'Ura - Concentration uranium',
    'Clim - Durée ensoleillement',
    'Clim - Précipitation',
    'Clim - Température',
    'Relief - Altitude (m)',
    'Rdn Map - Probabilité dépassement valeur réf radon',
    'Rdn Map - Index de confiance dépassement valeur réf',
    #'distance_to_stats',
    #'HydroWM -  Area',
    #'TBB - FLAECHE',
    'Geol - Distance to fault (m)',
    'Coordonnée E',
    'Coordonnée N',
   # 'Rdn - Début_year',
   # 'Rdn - Début_month',
   # 'Rdn - Début_day',
   # 'Rdn - Fin_year',
   # 'Rdn - Fin_month',
   # 'Rdn - Fin_day',
   # 'Rdn - Etage',
    
]

categorical_features = [
    #'Rdn - Canton',
    #'Rdn - Commune',
    'Rdn - Catégorie de bâtiment',
    'Rdn - Fondation',
    #'Rdn - Protocole de mesure',  # not in the final dataset
    #'Rdn - Type de mesure',  # not in the final dataset
    #'Rdn - Type instrument de mesure',  # not in the final dataset
    'Rdn - Type de local',
    #'Rdn - Séjour de personnes',
    #'RegBL - Catégorie de bâtiment',  # not in the final dataset
    'RegBL - Classe de bâtiment',
    'RegBL - Générateur de chaleur pour le chauffage 1',
    # 'RegBL - Source d’énergie / de chaleur pour le chauffage 1',
    # "RegBL - Générateur de chaleur pour l'eau chaude 1",
    # "RegBL - Source d’énergie / de chaleur pour l'eau chaude 1",
    'UncSoil - Classe de sol',
    'HydroWM -  GWKName',
    'HydroWM -  Grundwasserleitertyp',
    'HydroWM -  Naturraum',
    'HydroWM -  Grundwasserregime',
    'Seism - ZONE',
    'Geol - LITHO',
    'Geol - PERIODE',
    'Geol - EPOQUE',
    'Geol - LEG_TEC_1',
    'Geol - LEG_TEC_2',
    'Geol - LEG_TEC_3',
    'Geol - AQUIFERE',
    'Geol - HYDROGEOL',
    'Geol - LITH_PET',
    'Geol - GENESE',
    'Geol - TYPE_ROCHE',
    'Forest - Region_fr',
    #'Forest - Region_en',
    # 'TBB - CH_CODE_HN',
    # 'TBB - CH_BEZ_F',
    # 'TBB - NAME',
    'Geol - ETAGE',
    #'RegBL - Période de construction'
]

In [ ]:
categorical_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('to_str', FunctionTransformer(lambda X: pd.DataFrame(X).astype(str), feature_names_out='one-to-one')),
    ('encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan)),
])
hgb_preprocessor = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numerical_features),
    ('cat', categorical_pipeline, categorical_features),
], verbose_feature_names_out=False)

cardinality = X_train[categorical_features].nunique()
categorical_mask = [False] * len(numerical_features) + [cardinality[col] <= 255 for col in categorical_features]
hgb = Pipeline([
    ('prep', hgb_preprocessor),
    ('hgb', HistGradientBoostingRegressor(categorical_features=categorical_mask, random_state=RANDOM_SEED)),
])
hgb.fit(X_train, np.log1p(y_train))
evaluate('Hist gradient boosting', hgb.predict(X_test))

## Leaderboard

In [ ]:
pd.DataFrame(results).sort_values('R2 (log)', ascending=False).reset_index(drop=True)